In diesem Skript werden die Sentinel-5P Daten (NO2-column & UVAI) & ERA5 Daten (PBLH) gedownloadet.

In [1]:
import ee
import os
import requests
import pandas as pd

# =========================================================
# 1. Earth Engine Initialisierung
# =========================================================
print("Initialisiere Google Earth Engine...")
ee.Initialize(project='ee-arthurdreher01')

# =========================================================
# 2. Einstellungen & Pfade
# =========================================================
BASE_DIR = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)"

# Drei neue Unterordner für unsere drei Features anlegen
OUT_NO2 = os.path.join(BASE_DIR, "S5P_NO2")
OUT_UVAI = os.path.join(BASE_DIR, "S5P_UVAI")
OUT_BLH = os.path.join(BASE_DIR, "ERA5_BLH")

os.makedirs(OUT_NO2, exist_ok=True)
os.makedirs(OUT_UVAI, exist_ok=True)
os.makedirs(OUT_BLH, exist_ok=True)

START_DATE = "2024-05-01"
END_DATE = "2024-08-31"

# Bounding Box für Würzburg
LON_MIN, LON_MAX = 9.85, 10.05
LAT_MIN, LAT_MAX = 49.74, 49.85
region = ee.Geometry.BBox(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)

# =========================================================
# 3. Download-Funktion definieren
# =========================================================
def download_image(image, folder_path, filename):
    """Hilfsfunktion für den GEE Download nach EPSG:25832"""
    try:
        url = image.getDownloadURL({
            'scale': 1000,          # 1km Pixelgröße als robuster Standard
            'crs': 'EPSG:25832',    # Transformation ins Würzburg-Metriksystem
            'region': region,
            'format': 'GEO_TIFF'
        })
        response = requests.get(url)
        with open(os.path.join(folder_path, filename), 'wb') as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"    -> Fehler bei {filename}: {e}")
        return False

# =========================================================
# 4. Tägliche Iteration & Aggregation
# =========================================================
days = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
print(f"Starte Download-Prozess für {len(days)} Tage...\n")

for date in days:
    date_str = date.strftime("%Y-%m-%d")
    next_day_str = (date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    print(f"[{date_str}] Verarbeite Daten...")

    # ---------------------------------------------------------
    # A) Sentinel-5P NO2 (Tages-Mittelwert)
    # ---------------------------------------------------------
    no2_coll = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                .filterDate(date_str, next_day_str)
                .filterBounds(region)
                .select('tropospheric_NO2_column_number_density'))

    if no2_coll.size().getInfo() > 0:
        download_image(no2_coll.mean().clip(region), OUT_NO2, f"S5P_NO2_{date_str}.tif")
    else:
        print("    -> Keine NO2 Daten (Wolken/Satellitenlücke).")

    # ---------------------------------------------------------
    # B) Sentinel-5P UVAI (Tages-Mittelwert)
    # ---------------------------------------------------------
    uvai_coll = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_AER_AI')
                 .filterDate(date_str, next_day_str)
                 .filterBounds(region)
                 .select('absorbing_aerosol_index'))

    if uvai_coll.size().getInfo() > 0:
        download_image(uvai_coll.mean().clip(region), OUT_UVAI, f"S5P_UVAI_{date_str}.tif")
    else:
        print("    -> Keine UVAI Daten.")

    # ---------------------------------------------------------
    # C) ERA5 Boundary Layer Height (13:00 Uhr Snapshot)
    # ---------------------------------------------------------
    # Für ERA5 filtern wir ganz exakt auf 12:00 bis 14:00 Uhr, um den
    # Zustand der Grenzschicht exakt zum Überflug von S5P abzugreifen.
    start_time = f"{date_str}T12:00:00"
    end_time = f"{date_str}T14:00:00"

    blh_coll = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
                .filterDate(start_time, end_time)
                .filterBounds(region)
                .select('boundary_layer_height'))

    if blh_coll.size().getInfo() > 0:
        # Kein Skalierungsfaktor nötig, ERA5 BLH kommt direkt in Metern.
        download_image(blh_coll.mean().clip(region), OUT_BLH, f"ERA5_BLH_{date_str}.tif")
    else:
        print("    -> Keine ERA5 BLH Daten gefunden.")

print("\n" + "-" * 60)
print(f"DOWNLOAD ABGESCHLOSSEN! Alle Rasterdateien liegen sicher in den jeweiligen Unterordnern unter:\n{BASE_DIR}")
print("-" * 60)

Initialisiere Google Earth Engine...
Starte Download-Prozess für 123 Tage...

[2024-05-01] Verarbeite Daten...
[2024-05-02] Verarbeite Daten...
[2024-05-03] Verarbeite Daten...
[2024-05-04] Verarbeite Daten...
[2024-05-05] Verarbeite Daten...
[2024-05-06] Verarbeite Daten...
[2024-05-07] Verarbeite Daten...
[2024-05-08] Verarbeite Daten...
[2024-05-09] Verarbeite Daten...
[2024-05-10] Verarbeite Daten...
[2024-05-11] Verarbeite Daten...
[2024-05-12] Verarbeite Daten...
[2024-05-13] Verarbeite Daten...
[2024-05-14] Verarbeite Daten...
[2024-05-15] Verarbeite Daten...
[2024-05-16] Verarbeite Daten...
[2024-05-17] Verarbeite Daten...
[2024-05-18] Verarbeite Daten...
[2024-05-19] Verarbeite Daten...
[2024-05-20] Verarbeite Daten...
[2024-05-21] Verarbeite Daten...
[2024-05-22] Verarbeite Daten...
[2024-05-23] Verarbeite Daten...
[2024-05-24] Verarbeite Daten...
[2024-05-25] Verarbeite Daten...
[2024-05-26] Verarbeite Daten...
[2024-05-27] Verarbeite Daten...
[2024-05-28] Verarbeite Daten..